In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# paper-main-reconstruction-v2 — 全部论文结果总入口

默认仅运行 **main 的小规模真实预检**：新生成一对图，覆盖6种条件，显式检查连续预测H及post，并做一次评分复用A/B。不会启动2000/3000/1000全量任务；`science_denominator=0`。GPU型号与版本只记录，不限A100。

每次只选择一个worker。可在不同runtime独立运行main和四个baseline；重建等待main的新阈值和前100评估配对，最后运行finalize。所有输出位于`MyDrive/CEG-WM/paper-main-reconstruction-v2`，各worker有独立子目录。旧v1和reflect结果保留。

正式范围：五方法各2000校准、3000独立clean负样本、1000配对×6条件；main前100配对四消融；五方法PSNR/SSIM/LPIPS；SDXL前100配对重建；统一JSON、60行主表、83行完整二元结果、15行画质结果及PNG/PDF。六条件仍是clean、JPEG50、resize .5、crop .8、blur1及+10°旋转；旋转统一改为黑填充双线性。

逐格查看输出。需要正式运行时，明确将MODE改为formal；不要用Run all意外跳过检查或反复切换多个模型。预检不按检出率设通过门槛；正式FPR/CI报告但不阻止产出。

In [ ]:
WORKER = 'main' # @param ['main', 't2smark', 'tree_ring', 'gaussian_shading', 'shallow_diffuse', 'reconstruction', 'finalize']
MODE = 'preflight' # @param ['preflight', 'formal']
print({'worker':WORKER,'mode':MODE,'only_one_worker':True})

In [ ]:
from pathlib import Path
import subprocess, sys, os
EXACT='bc92f4332a2fa0ea90fd40a12a25a035e6decdad'
REPO=Path('/content/ceg-wm-paper-v2-bc92f43')
DRIVE_ROOT=Path('/content/drive/MyDrive/CEG-WM')
RUNTIME_ROOT=Path('/content/paper-v2-runtime')
if not REPO.exists():
    subprocess.run(['git','clone','--branch','main','--single-branch','https://github.com/RICHAAARC/CEG-WM.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach',EXACT],check=True)
if WORKER in ('t2smark','tree_ring','gaussian_shading','shallow_diffuse'):
    dependencies=['diffusers==0.32.0','transformers==4.45.2','accelerate==1.1.1','huggingface_hub==0.26.2','safetensors==0.4.5','sentencepiece==0.2.0','lpips','torchmetrics','matplotlib']
else:
    dependencies=['diffusers<0.40','transformers','accelerate','lpips','torchmetrics','matplotlib']
subprocess.run([sys.executable,'-m','pip','install','-q',*dependencies,str(REPO)],check=True)
if WORKER != 'finalize':
    from google.colab import userdata
    os.environ['HF_TOKEN']=userdata.get('HF_TOKEN')
    if WORKER in ('main','reconstruction'):
        os.environ['CEG_WM_ROOT_KEY']=userdata.get('CEG_WM_ROOT_KEY')
print({'code':EXACT,'experiment':'paper-main-reconstruction-v2','worker':WORKER,'mode':MODE})

运行下面一格只会执行所选worker和mode。预检输出含实测初始化、生成和逐条件评分耗时；失败预检可在修复后重跑，旧失败会单独保留。正式worker保留原有断点和固定失败行。

In [ ]:
command=[sys.executable,'-m','experiments.run_paper_v2','--worker',WORKER,'--mode',MODE,
         '--drive-root',str(DRIVE_ROOT),'--runtime-root',str(RUNTIME_ROOT)]
subprocess.run(command,cwd=REPO,check=True)
print('输出目录:', DRIVE_ROOT/'paper-main-reconstruction-v2')